# ЛР-02: Топливо для частей

## Student notebook: military 01

Этот notebook предназначен для самостоятельного решения.

Готового оптимального плана и заполненного решателя здесь нет.

## 1. Зачем нужен этот кейс

Три базы снабжения распределяют топливо между четырьмя частями.

После этого notebook-а студент должен уметь:

1. читать военный логистический сценарий как обычную транспортную модель;
2. не путать физический объём топлива и стоимость маршрута;
3. проверять допустимость плана по каждой базе и каждой части;
4. интерпретировать, почему solver выбирает именно эти направления.

## 2. Исходные данные

Тип задачи: **закрытая**.

### Запасы

| Поставщик | Объём |
| --- | --- |
| База A | 50 |
| База B | 40 |
| База C | 35 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Часть 1 | 30 |
| Часть 2 | 25 |
| Часть 3 | 35 |
| Часть 4 | 35 |

### Матрица затрат

| Откуда / Куда | Часть 1 | Часть 2 | Часть 3 | Часть 4 |
| --- | --- | --- | --- | --- |
| База A | 6 | 7 | 9 | 12 |
| База B | 5 | 4 | 8 | 10 |
| База C | 8 | 6 | 5 | 7 |

## 3. Что нужно сделать

Идите тем же маршрутом, что и в теории ЛР-02:

1. Проверьте баланс запасов и спроса.
2. Если задача открытая, добавьте фиктивный узел и поясните его смысл.
3. Запишите математическую модель через переменные $x_{ij}$.
4. Сформируйте `c`, `A_eq`, `b_eq`, `bounds` для `linprog`.
5. Проверьте, что `A_eq x = b_eq` действительно задаёт строки и столбцы.
6. После собственной попытки решите задачу через Python.
7. Кратко объясните реальные и фиктивные маршруты.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog


DUMMY_SUPPLIER_NAME = "Фиктивный поставщик"
DUMMY_CONSUMER_NAME = "Фиктивный потребитель"
BALANCE_TOLERANCE = 1e-9

# Шаг 1: записываем данные задачи, не меняя числовую постановку.
supplier_names = [
    'База A',
    'База B',
    'База C',
]
consumer_names = [
    'Часть 1',
    'Часть 2',
    'Часть 3',
    'Часть 4',
]

supplies = np.array(
    [
        50,
        40,
        35,
    ],
    dtype=float,
)

demands = np.array(
    [
        30,
        25,
        35,
        35,
    ],
    dtype=float,
)

costs = np.array(
    [
        [6, 7, 9, 12],
        [5, 4, 8, 10],
        [8, 6, 5, 7],
    ],
    dtype=float,
)

# Шаг 2: выводим векторы и матрицу затрат в читаемом виде.
supply_df = pd.DataFrame({"запас": supplies}, index=supplier_names)
demand_df = pd.DataFrame({"спрос": demands}, index=consumer_names)
cost_df = pd.DataFrame(costs, index=supplier_names, columns=consumer_names)

print("Запасы поставщиков a_i:")
display(supply_df)

print("Спрос потребителей b_j:")
display(demand_df)

print("Матрица затрат c_ij:")
display(cost_df)

print("sum supply =", supplies.sum())
print("sum demand =", demands.sum())


Запасы поставщиков a_i:


,запас
База A,50.0
База B,40.0
База C,35.0


Спрос потребителей b_j:


,спрос
Часть 1,30.0
Часть 2,25.0
Часть 3,35.0
Часть 4,35.0


Матрица затрат c_ij:


,Часть 1,Часть 2,Часть 3,Часть 4
База A,6.0,7.0,9.0,12.0
База B,5.0,4.0,8.0,10.0
База C,8.0,6.0,5.0,7.0


sum supply = 125.0
sum demand = 125.0


## 4. Шаблон для самостоятельной сборки модели

Ниже намеренно оставлены `TODO`. Это не готовое решение, а guided skeleton:
он показывает правильные имена, порядок шагов и форму LP-модели, но ключевые
строки вы заполняете самостоятельно.


In [2]:
def balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
):
    """Готовит закрытую транспортную модель через фиктивный узел."""

    BALANCE_TOLERANCE = 1e-9

    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    # Добавляем фиктивного потребителя, если запасов больше
    if balance_difference > BALANCE_TOLERANCE:
        # Добавляем фиктивного потребителя
        balanced_demands = np.append(balanced_demands, balance_difference)
        balanced_consumer_names.append(DUMMY_CONSUMER_NAME)

        # Добавляем столбец с нулевыми затратами для фиктивного потребителя
        dummy_column = np.full((balanced_costs.shape[0], 1), dummy_cost)
        balanced_costs = np.column_stack([balanced_costs, dummy_column])

        balance_note = f"Задача открытая: запасов больше на {balance_difference:.1f}. Добавлен фиктивный потребитель."

    # Добавляем фиктивного поставщика, если спроса больше
    elif balance_difference < -BALANCE_TOLERANCE:
        # Добавляем фиктивного поставщика
        balanced_supplies = np.append(balanced_supplies, -balance_difference)
        balanced_supplier_names.append(DUMMY_SUPPLIER_NAME)

        # Добавляем строку с нулевыми затратами для фиктивного поставщика
        dummy_row = np.full((1, balanced_costs.shape[1]), dummy_cost)
        balanced_costs = np.vstack([balanced_costs, dummy_row])

        balance_note = f"Задача открытая: спроса больше на {-balance_difference:.1f}. Добавлен фиктивный поставщик."

    else:
        balance_note = "Задача закрытая: сумма запасов = сумме спроса"

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )

In [3]:
def build_transport_lp(supplies, demands, costs):
    """Собирает c, A_eq, b_eq и bounds для linprog."""

    supplier_count, consumer_count = costs.shape
    variable_count = supplier_count * consumer_count

    # Вектор цели (стоимости перевозок)
    c = costs.flatten()

    # Матрица ограничений A_eq
    A_eq = np.zeros((supplier_count + consumer_count, variable_count))

    # Заполняем строки поставщиков
    for i in range(supplier_count):
        for j in range(consumer_count):
            idx = i * consumer_count + j
            A_eq[i, idx] = 1

    # Заполняем строки потребителей
    for j in range(consumer_count):
        for i in range(supplier_count):
            idx = i * consumer_count + j
            A_eq[supplier_count + j, idx] = 1

    # Вектор правых частей
    b_eq = np.concatenate([supplies, demands])

    # Границы переменных
    bounds = [(0.0, None) for _ in range(variable_count)]

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
        "route_index_hint": "supplier_idx * consumer_count + consumer_idx",
        "variable_count": variable_count,
    }

In [4]:
# Шаг 1: сбалансируйте задачу
(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
    balanced_supplier_names,
    balanced_consumer_names,
    balance_note,
) = balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
)

print(balance_note)
print("balanced supply =", balanced_supplies.sum())
print("balanced demand =", balanced_demands.sum())
print("\nСбалансированные данные:")
print("Запасы:", balanced_supplies)
print("Спрос:", balanced_demands)

# Проверка баланса
assert np.allclose(balanced_supplies.sum(), balanced_demands.sum())

# Шаг 2: соберите LP-модель
lp_model = build_transport_lp(balanced_supplies, balanced_demands, balanced_costs)

# Вывод переменных и их стоимости
route_labels = [
    f"x_{supplier_idx + 1},{consumer_idx + 1}"
    for supplier_idx in range(len(balanced_supplier_names))
    for consumer_idx in range(len(balanced_consumer_names))
]

print("\nМатрица затрат (с фиктивным потребителем):")
balanced_cost_df = pd.DataFrame(
    balanced_costs,
    index=balanced_supplier_names,
    columns=balanced_consumer_names
)
display(balanced_cost_df)

print("\nПеременные и их стоимости:")
display(pd.DataFrame({"переменная": route_labels, "стоимость c": lp_model["c"]}))

print("\nМатрица ограничений A_eq:")
display(pd.DataFrame(lp_model["A_eq"], columns=route_labels))

print("\nПравые части ограничений b_eq:")
display(pd.DataFrame({"b_eq": lp_model["b_eq"]}))

Задача закрытая: сумма запасов = сумме спроса
balanced supply = 125.0
balanced demand = 125.0

Сбалансированные данные:
Запасы: [50. 40. 35.]
Спрос: [30. 25. 35. 35.]

Матрица затрат (с фиктивным потребителем):


,Часть 1,Часть 2,Часть 3,Часть 4
База A,6.0,7.0,9.0,12.0
База B,5.0,4.0,8.0,10.0
База C,8.0,6.0,5.0,7.0



Переменные и их стоимости:


,переменная,стоимость c
0,"x_1,1",6.0
1,"x_1,2",7.0
2,"x_1,3",9.0
3,"x_1,4",12.0
4,"x_2,1",5.0
5,"x_2,2",4.0
6,"x_2,3",8.0
7,"x_2,4",10.0
8,"x_3,1",8.0
9,"x_3,2",6.0



Матрица ограничений A_eq:


,"x_1,1","x_1,2","x_1,3","x_1,4","x_2,1","x_2,2","x_2,3","x_2,4","x_3,1","x_3,2","x_3,3","x_3,4"
0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
6,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0



Правые части ограничений b_eq:


,b_eq
0,50.0
1,40.0
2,35.0
3,30.0
4,25.0
5,35.0
6,35.0


In [5]:
# Решение задачи
result = linprog(
    lp_model["c"],
    A_eq=lp_model["A_eq"],
    b_eq=lp_model["b_eq"],
    bounds=lp_model["bounds"],
    method="highs",
)

print(f"Успех решения: {result.success}")
print(f"Сообщение: {result.message}")

if result.success:
    # Преобразуем решение в матрицу плана перевозок
    plan = result.x.reshape(len(balanced_supplier_names), len(balanced_consumer_names))
    plan_df = pd.DataFrame(
        plan,
        index=balanced_supplier_names,
        columns=balanced_consumer_names,
        dtype=float
    )

    print("\nОптимальный план перевозок:")
    display(plan_df.round(2))

    # Добавим итоговые строки и столбцы
    plan_with_totals = plan_df.round(2).copy()
    plan_with_totals.loc['Итого'] = plan_with_totals.sum(axis=0)
    plan_with_totals['Итого'] = plan_with_totals.sum(axis=1)

    print("\nПлан перевозок с итогами:")
    display(plan_with_totals)

    # Проверка сумм по строкам и столбцам
    print("\nПроверка ограничений:")
    print("Строки (поставщики):")
    for i, name in enumerate(balanced_supplier_names):
        row_sum = plan[i, :].sum()
        print(f"  {name}: {row_sum:.1f} = {balanced_supplies[i]:.1f} ✓")

    print("Столбцы (потребители):")
    for j, name in enumerate(balanced_consumer_names):
        col_sum = plan[:, j].sum()
        print(f"  {name}: {col_sum:.1f} = {balanced_demands[j]:.1f} ✓")

    # Общая стоимость
    total_cost = result.fun
    print(f"\nМинимальная общая стоимость перевозок: {total_cost:.2f}")

    # Проверка через assert
    assert np.allclose(plan.sum(axis=1), balanced_supplies)
    assert np.allclose(plan.sum(axis=0), balanced_demands)
    print("\nВсе проверки пройдены успешно!")

    # Сохраняем план для анализа
    plan_matrix = plan

else:
    print("Решение не найдено!")
    plan_matrix = None

Успех решения: True
Сообщение: Optimization terminated successfully. (HiGHS Status 7: Optimal)

Оптимальный план перевозок:


,Часть 1,Часть 2,Часть 3,Часть 4
База A,15.0,0.0,35.0,0.0
База B,15.0,25.0,0.0,0.0
База C,0.0,0.0,-0.0,35.0



План перевозок с итогами:


,Часть 1,Часть 2,Часть 3,Часть 4,Итого
База A,15.0,0.0,35.0,0.0,50.0
База B,15.0,25.0,0.0,0.0,40.0
База C,0.0,0.0,-0.0,35.0,35.0
Итого,30.0,25.0,35.0,35.0,125.0



Проверка ограничений:
Строки (поставщики):
  База A: 50.0 = 50.0 ✓
  База B: 40.0 = 40.0 ✓
  База C: 35.0 = 35.0 ✓
Столбцы (потребители):
  Часть 1: 30.0 = 30.0 ✓
  Часть 2: 25.0 = 25.0 ✓
  Часть 3: 35.0 = 35.0 ✓
  Часть 4: 35.0 = 35.0 ✓

Минимальная общая стоимость перевозок: 825.00

Все проверки пройдены успешно!


In [7]:
# Анализ оптимальных маршрутов
print("\n АНАЛИЗ ОПТИМАЛЬНЫХ МАРШРУТОВ \n")

if plan_matrix is not None:
    # Отделяем реальные и фиктивные маршруты
    real_consumer_count = len(consumer_names)
    dummy_col_index = real_consumer_count

    print("Активные реальные маршруты (ненулевые перевозки):")
    active_routes = []
    for i, supplier in enumerate(balanced_supplier_names):
        if i >= len(supplier_names):  # пропускаем фиктивного поставщика
            continue
        for j, consumer in enumerate(balanced_consumer_names):
            if j >= real_consumer_count:  # пропускаем фиктивного потребителя
                continue
            amount = plan_matrix[i, j]
            if amount > 0.01:
                cost = costs[i, j]
                total = amount * cost
                active_routes.append({
                    'Маршрут': f"{supplier} → {consumer}",
                    'Объем': f"{amount:.1f}",
                    'Стоимость за ед.': f"{cost:.0f}",
                    'Общая стоимость': f"{total:.1f}"
                })

    if active_routes:
        active_df = pd.DataFrame(active_routes)
        display(active_df)
    else:
        print("Нет активных маршрутов")

    # Проверяем наличие фиктивного потребителя
    print(f"\nКоличество реальных потребителей: {real_consumer_count}")
    print(f"Количество всех потребителей в модели: {len(balanced_consumer_names)}")
    print(f"Индекс фиктивного потребителя: {dummy_col_index}")

    if dummy_col_index < plan_matrix.shape[1]:
        print("\nНераспределенное питание (фиктивные маршруты):")
        dummy_routes = []
        dummy_amounts = []
        for i, supplier in enumerate(balanced_supplier_names):
            if i >= len(supplier_names):
                continue
            amount = plan_matrix[i, dummy_col_index]
            dummy_amounts.append(amount)
            if amount > 0.01:
                dummy_routes.append({
                    'Маршрут': f"{supplier} → {DUMMY_CONSUMER_NAME}",
                    'Объем': f"{amount:.1f}",
                    'Причина': 'Избыток продукции'
                })

        if dummy_routes:
            dummy_df = pd.DataFrame(dummy_routes)
            display(dummy_df)
        else:
            print("Нет фиктивных маршрутов (все запасы использованы)")
    else:
        print("\nФиктивный потребитель не добавлен (задача закрыта)")
        dummy_amounts = []

    # Общая стоимость по реальным маршрутам
    if active_routes:
        real_total = sum(float(r['Общая стоимость']) for r in active_routes)
        print(f"\nОбщая стоимость реальных перевозок: {real_total:.1f}")
        print(f"Общая стоимость перевозок (включая фиктивные): {result.fun:.1f}")
        print(f"Совпадение: {'✓' if abs(real_total - result.fun) < 0.01 else '✗'}")
    else:
        print("\nНет активных маршрутов для расчета стоимости")

    print("\n ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТА ")
    if dummy_amounts:
        total_dummy = sum(dummy_amounts)
        print(f"Общий избыток питания составляет {total_dummy:.1f} единиц")
        print("Это означает, что часть продукции останется на складах")
        print("и не будет доставлена в школы.")
    else:
        print("Все запасы полностью распределены")

    # Дополнительная интерпретация
    print("\nДетальная интерпретация:")
    if dummy_amounts:
        for i, supplier in enumerate(balanced_supplier_names):
            if i >= len(supplier_names):
                continue
            if i < len(dummy_amounts) and dummy_amounts[i] > 0.01:
                print(f"  • {supplier}: {dummy_amounts[i]:.1f} единиц не будет отправлено")

    print("\nНаиболее загруженные маршруты (по объему):")
    if active_routes:
        sorted_routes = sorted(active_routes, key=lambda x: float(x['Объем']), reverse=True)
        for route in sorted_routes[:3]:
            print(f"  • {route['Маршрут']}: {route['Объем']} единиц, стоимость {route['Общая стоимость']}")

    print("\nНаиболее дорогие маршруты (по общей стоимости):")
    if active_routes:
        sorted_by_cost = sorted(active_routes, key=lambda x: float(x['Общая стоимость']), reverse=True)
        for route in sorted_by_cost[:3]:
            print(f"  • {route['Маршрут']}: {route['Общая стоимость']}, объем {route['Объем']} единиц")

else:
    print("Нет решения для анализа!")


 АНАЛИЗ ОПТИМАЛЬНЫХ МАРШРУТОВ 

Активные реальные маршруты (ненулевые перевозки):


,Маршрут,Объем,Стоимость за ед.,Общая стоимость
0,База A → Часть 1,15.0,6,90.0
1,База A → Часть 3,35.0,9,315.0
2,База B → Часть 1,15.0,5,75.0
3,База B → Часть 2,25.0,4,100.0
4,База C → Часть 4,35.0,7,245.0



Количество реальных потребителей: 4
Количество всех потребителей в модели: 4
Индекс фиктивного потребителя: 4

Фиктивный потребитель не добавлен (задача закрыта)

Общая стоимость реальных перевозок: 825.0
Общая стоимость перевозок (включая фиктивные): 825.0
Совпадение: ✓

 ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТА 
Все запасы полностью распределены

Детальная интерпретация:

Наиболее загруженные маршруты (по объему):
  • База A → Часть 3: 35.0 единиц, стоимость 315.0
  • База C → Часть 4: 35.0 единиц, стоимость 245.0
  • База B → Часть 2: 25.0 единиц, стоимость 100.0

Наиболее дорогие маршруты (по общей стоимости):
  • База A → Часть 3: 315.0, объем 35.0 единиц
  • База C → Часть 4: 245.0, объем 35.0 единиц
  • База B → Часть 2: 100.0, объем 25.0 единиц


## 5. Что должно быть в отчёте

1. Таблица исходных данных и проверка баланса.
2. Полная математическая постановка через $x_{ij}$.
3. Пояснение, нужен ли фиктивный поставщик или фиктивный потребитель.
4. Векторно-матричная запись `c`, `A_eq`, `b_eq`, `bounds`.
5. Проверка через `linprog`.
6. Таблица оптимального плана перевозок.
7. Проверка сумм по строкам и столбцам.
8. Содержательная интерпретация ненулевых маршрутов.

## 6. Контрольный чек-лист

- [ ] Я сам проверил баланс до запуска solver-а.
- [ ] Я осмысленно собрал `A_eq` и `b_eq` через индекс `route_index`.
- [ ] Я проверил `bounds` и неотрицательность всех $x_{ij}$.
- [ ] Я отличаю реальные маршруты от фиктивного узла.
- [ ] Я объяснил полученный план простыми словами.
